In [ ]:
#####################################################################
# STEP 1: IMPORTING LIBRARIES & HARDWARE CHECK
#####################################################################
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['NCCL_DEBUG'] = 'WARN'
print("[INFO] Loading required python libraries...")
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from glob import glob
from PIL import Image

import tensorflow as tf
print("\n" + "="*50)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("Name: ", tf.config.list_physical_devices('GPU'))
print("="*50 + "\n")

from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Recall, Precision
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, LeakyReLU, Add, Multiply
from tensorflow.keras.models import Model

#####################################################################
# STEP 2: METRICS, LOSS FUNCTIONS & CONSTANTS
#####################################################################
print("[INFO] Defining architecture constants, metrics, and loss functions...")

# ── Configuration ────────────────────────────────────────────────
H = 256
W = 256
BATCH_SIZE_PER_REPLICA = 8
LEARNING_RATE = 1e-4
EPOCHS = 150
PATIENCE = 10               # EarlyStopping patience
LR_PATIENCE = 5             # FIX: ReduceLROnPlateau fires BEFORE early stopping
USE_AUGMENTATION = True     # FIX: clean boolean flag instead of hardcoded `if True`
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
# ─────────────────────────────────────────────────────────────────

smooth = 1e-6  # FIX: was 1e-15 at module level but never used (shadowed inside functions)

def soft_dice_coef(y_true, y_pred):
    intersection = tf.reduce_sum(y_true * y_pred)
    denominator = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return (2. * intersection + smooth) / (denominator + smooth)

def soft_dice_loss(y_true, y_pred):
    return 1.0 - soft_dice_coef(y_true, y_pred)

def combo_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = soft_dice_loss(y_true, y_pred)
    return bce + dice

#####################################################################
# STEP 3: DATA PREPROCESSING PIPELINE
#####################################################################

def load_data(data_path):
    print(f"[INFO] Scanning for data inside: {data_path} ...")
    if not os.path.exists(data_path):
        print(f"[ERROR] CRITICAL FAILURE: Directory {data_path} does not exist!")
        return ([], []), ([], []), ([], [])

    images = sorted(glob(os.path.join(data_path, "*_sat.jpg")))
    masks  = sorted(glob(os.path.join(data_path, "*_mask.png")))

    if len(images) == 0:
        print(f"[ERROR] CRITICAL FAILURE: No '*_sat.jpg' files found.")
        return ([], []), ([], []), ([], [])

    # FIX: assert image/mask counts match to catch misaligned datasets early
    assert len(images) == len(masks), \
        f"[ERROR] Mismatch: {len(images)} images vs {len(masks)} masks!"

    print(f"[INFO] Successfully located {len(images)} image-mask pairs.")

    # Clean 80/10/10 split
    train_x, temp_x, train_y, temp_y = train_test_split(images, masks, test_size=0.2, random_state=42)
    val_x, test_x, val_y, test_y     = train_test_split(temp_x, temp_y, test_size=0.5, random_state=42)

    return (train_x, train_y), (val_x, val_y), (test_x, test_y)

def read_image(path):
    try:
        img = Image.open(path).convert('RGB')  # FIX: force RGB to avoid RGBA crash on some satellite tiles
        img = img.resize((W, H))
        return np.array(img, dtype=np.float32) / 255.0
    except Exception as e:
        print(f"[WARNING] Corrupted image skipped: {path} | {e}")
        return None

def read_mask(path):
    try:
        mask = Image.open(path).convert('L')
        mask = mask.resize((W, H))
        x = np.array(mask, dtype=np.float32) / 255.0
        return np.expand_dims(x, axis=-1)
    except Exception as e:
        print(f"[WARNING] Corrupted mask skipped: {path} | {e}")
        return None

def tf_parse(x, y):
    def _parse(x, y):
        return read_image(x), read_mask(y)
    x, y = tf.numpy_function(_parse, [x, y], [tf.float32, tf.float32])
    x.set_shape([H, W, 3]); y.set_shape([H, W, 1])
    return x, y

def heavy_augment(x, y):
    """Geometric + photometric augmentation applied only to training data."""
    if tf.random.uniform(()) > 0.5: x = tf.image.flip_left_right(x);  y = tf.image.flip_left_right(y)
    if tf.random.uniform(()) > 0.5: x = tf.image.flip_up_down(x);     y = tf.image.flip_up_down(y)
    if tf.random.uniform(()) > 0.5:
        x = tf.image.random_brightness(x, max_delta=0.2)
        x = tf.image.random_contrast(x, lower=0.8, upper=1.2)
    x = tf.clip_by_value(x, 0.0, 1.0)  # FIX: clamp after brightness/contrast to avoid out-of-range values
    return x, y

#####################################################################
# STEP 4: MODEL ARCHITECTURE (ATTENTION-GUIDED RESNET U-NET)
#####################################################################

def conv_block(x, filters, kernel_size=3, padding='same'):
    x = Conv2D(filters, kernel_size, padding=padding)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    x = Conv2D(filters, kernel_size, padding=padding)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(negative_slope=0.1)(x)
    return x

def residual_block(x, filters):
    """1×1 projection shortcut + double conv, then residual Add."""
    res = Conv2D(filters, (1, 1), padding='same')(x)
    res = BatchNormalization()(res)
    x   = conv_block(x, filters)
    x   = Add()([x, res])
    x   = LeakyReLU(negative_slope=0.1)(x)
    return x

def attention_gate(x, g, inter_filters):
    """Attention Gate -- suppress irrelevant features in skip connections.
    x: encoder skip features, g: decoder gating signal, inter_filters: bottleneck channels.
    Returns: x weighted by learned spatial attention coefficients."""
    Wg = Conv2D(inter_filters, (1, 1), padding='same')(g)
    Wg = BatchNormalization()(Wg)
    Wx = Conv2D(inter_filters, (1, 1), padding='same')(x)
    Wx = BatchNormalization()(Wx)
    psi = Add()([Wg, Wx])
    psi = LeakyReLU(negative_slope=0.1)(psi)
    psi = Conv2D(1, (1, 1), padding='same', activation='sigmoid')(psi)
    return Multiply()([x, psi])

def build_resnet(input_shape=(256, 256, 3)):
    inputs = Input(input_shape)

    # Encoder
    c1 = residual_block(inputs, 64);  p1 = MaxPool2D((2, 2))(c1)
    c2 = residual_block(p1, 128);     p2 = MaxPool2D((2, 2))(c2)
    c3 = residual_block(p2, 256);     p3 = MaxPool2D((2, 2))(c3)
    c4 = residual_block(p3, 512);     p4 = MaxPool2D((2, 2))(c4)

    # Bottleneck
    bn = residual_block(p4, 1024)

    # Decoder
    d1 = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(bn)
    c4_att = attention_gate(c4, d1, 256)
    d1 = Concatenate()([d1, c4_att]); d1 = residual_block(d1, 512)
    d2 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(d1)
    c3_att = attention_gate(c3, d2, 128)
    d2 = Concatenate()([d2, c3_att]); d2 = residual_block(d2, 256)
    d3 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(d2)
    c2_att = attention_gate(c2, d3, 64)
    d3 = Concatenate()([d3, c2_att]); d3 = residual_block(d3, 128)
    d4 = Conv2DTranspose(64,  (2, 2), strides=(2, 2), padding='same')(d3)
    c1_att = attention_gate(c1, d4, 32)
    d4 = Concatenate()([d4, c1_att]); d4 = residual_block(d4, 64)

    outputs = Conv2D(1, (1, 1), padding='same', activation='sigmoid')(d4)
    return Model(inputs, outputs)

#####################################################################
# STEP 5: TRAINING
#####################################################################
DATASETS = [
    {
        "name": "DeepGlobe",
        "path": "/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/train"
    },
]

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for ds in DATASETS:
    print(f"\n{'='*50}")
    print(f"       STARTING TRAINING CYCLE: {ds['name']}")
    print(f"{'='*50}\n")

    # 1. Load Data
    (train_x, train_y), (val_x, val_y), (test_x, test_y) = load_data(ds['path'])
    if len(train_x) == 0:
        continue

    print(f"\n*Split Sizes*")
    print(f"Train: {len(train_x)} | Val: {len(val_x)} | Test: {len(test_x)}\n")

    # 2. Build tf.data pipelines
    strategy = tf.distribute.MirroredStrategy()
    GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync
    print(f"[INFO] MirroredStrategy active across {strategy.num_replicas_in_sync} device(s).")
    print(f"[INFO] Global Batch Size → {GLOBAL_BATCH_SIZE}")

    train_dataset = tf.data.Dataset.from_tensor_slices((train_x, train_y))
    train_dataset = train_dataset.map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)

    if USE_AUGMENTATION:  # FIX: clean config flag
        print(f"[INFO] Data Augmentation (heavy_augment) is ON.")
        train_dataset = train_dataset.map(heavy_augment, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        print(f"[INFO] Data Augmentation is OFF.")

    train_dataset = train_dataset.shuffle(buffer_size=500).repeat().batch(GLOBAL_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)  # FIX: added shuffle

    val_dataset = (tf.data.Dataset.from_tensor_slices((val_x, val_y))
                   .map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
                   .batch(GLOBAL_BATCH_SIZE)
                   .prefetch(tf.data.AUTOTUNE))

    # FIX: also build a test dataset for final evaluation
    test_dataset = (tf.data.Dataset.from_tensor_slices((test_x, test_y))
                    .map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
                    .batch(GLOBAL_BATCH_SIZE)
                    .prefetch(tf.data.AUTOTUNE))

    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
    train_dataset = train_dataset.with_options(options)
    val_dataset = val_dataset.with_options(options)
    test_dataset = test_dataset.with_options(options)

    train_steps = np.ceil(len(train_x) / GLOBAL_BATCH_SIZE).astype(int)

    # Quick shape sanity check
    for batch_idx, (xx, yy) in enumerate(train_dataset.take(1)):
        print(f"\nBatch shape check — X: {xx.shape} {xx.dtype} | Y: {yy.shape} {yy.dtype}\n")

    # 3. Build & Compile
    print(f"[INFO] Building ResNet U-Net inside strategy scope...")
    with strategy.scope():
        model = build_resnet(input_shape=(H, W, 3))

        def iou(y_true, y_pred):
            y_pred = tf.cast(y_pred > 0.5, tf.float32)
            intersection = tf.reduce_sum(y_true * y_pred)
            union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
            return (intersection + smooth) / (union + smooth)

        def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75):
            """
            Focal Tversky Loss — ideal for imbalanced road segmentation.
            alpha > beta: heavily penalizes false negatives (missed faint roads).
            gamma: focal parameter concentrating loss on hardest pixels.
            """
            y_true = tf.cast(y_true, tf.float32)
            y_pred = tf.cast(y_pred, tf.float32)
            tp = tf.reduce_sum(y_true * y_pred)
            fn = tf.reduce_sum(y_true * (1 - y_pred))
            fp = tf.reduce_sum((1 - y_true) * y_pred)
            tversky_index = (tp + smooth) / (tp + alpha * fn + beta * fp + smooth)
            return tf.pow((1 - tversky_index), gamma)

        def connectivity_penalty(y_true, y_pred):
            """Differentiable connectivity loss via Laplacian edge matching.
            Penalizes structural discontinuities in predicted road masks."""
            laplacian_kernel = tf.constant([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=tf.float32)
            laplacian_kernel = tf.reshape(laplacian_kernel, [3, 3, 1, 1])
            edges_pred = tf.nn.conv2d(y_pred, laplacian_kernel, strides=[1,1,1,1], padding='SAME')
            edges_true = tf.nn.conv2d(y_true, laplacian_kernel, strides=[1,1,1,1], padding='SAME')
            return tf.reduce_mean(tf.abs(edges_pred - edges_true))

        def proposed_loss(y_true, y_pred):
            """Novel Combined Loss: Focal Tversky + Connectivity Penalty (lambda=0.3).
            Focal Tversky handles class imbalance; connectivity penalty preserves road structure."""
            ftl = focal_tversky_loss(y_true, y_pred)
            conn = connectivity_penalty(y_true, y_pred)
            return ftl + 0.3 * conn

        model.compile(
            loss=proposed_loss,
            optimizer=Adam(LEARNING_RATE),
            metrics=[iou, Recall(), Precision()]
        )

    model.summary()

    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"best_{ds['name']}.keras")  # FIX: .keras not .h5
    callbacks = [
        # FIX: ModelCheckpoint — saves best weights mid-training, crash-safe
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_iou',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_iou',
            mode='max',
            patience=PATIENCE,          # 10
            restore_best_weights=True,
            verbose=1
        ),
        # FIX: LR_PATIENCE=5, fires well before EarlyStopping at 10
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_iou',
            factor=0.5,
            patience=LR_PATIENCE,
            min_lr=1e-6,
            verbose=1
        ),
    ]

    # 4. Train
    print("[INFO] Kicking off model.fit()...")
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        steps_per_epoch=train_steps,
        validation_data=val_dataset,
        callbacks=callbacks
    )

    # 5. Save final weights
    model_save_path = f"/kaggle/working/road_extraction_proposed_model_{ds['name']}.keras"  # FIX: .keras
    model.save(model_save_path)
    print(f"[INFO] Final model saved → {model_save_path}")

    # FIX: Evaluate on held-out test set (was missing entirely before)
    print("\n[INFO] Evaluating on held-out test set...")
    test_results = model.evaluate(test_dataset, verbose=1)
    metric_names = ['loss', 'iou', 'recall', 'precision']
    print("\n── Test Set Results ──")
    for name, val in zip(metric_names, test_results):
        print(f"  {name:>12}: {val:.4f}")

    # 6. Training curves
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'],     label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f"proposed_model ({ds['name']}) — Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['iou'],     label='Train IoU')
    plt.plot(history.history['val_iou'], label='Val IoU')
    plt.title(f"proposed_model ({ds['name']}) — IoU")
    plt.legend()
    plt.tight_layout()
    plt.show()

    #####################################################################
    # STEP 6: NOVEL IMPROVEMENT — FLOOD-FILL POST PROCESSING
    #####################################################################
    print(f"\n{'='*50}")
    print(f" EXECUTING FLOOD-FILL POST-PROCESSING ON TEST SAMPLE...")
    print(f"{'='*50}")

    sample_img = read_image(test_x[0])
    if sample_img is not None:
        prediction  = model.predict(np.expand_dims(sample_img, axis=0))[0]
        binary_mask = (prediction.squeeze() > 0.5).astype(np.uint8) * 255

        gray_original    = cv2.cvtColor(np.uint8(sample_img * 255), cv2.COLOR_RGB2GRAY)
        black_border_mask = (gray_original == 0).astype(np.uint8)

        h_img, w_img = binary_mask.shape
        clean_pred   = binary_mask.copy()

        # FIX: cv2.floodFill seed is (x, y) = (col, row), NOT numpy (row, col)
        # Corrected corner seeds in (col, row) / (x, y) order for cv2
        cv2_corners = [
            (0,         0        ),   # top-left
            (w_img - 1, 0        ),   # top-right
            (0,         h_img - 1),   # bottom-left
            (w_img - 1, h_img - 1),   # bottom-right
        ]
        for (cx, cy) in cv2_corners:
            numpy_row, numpy_col = cy, cx   # convert back to check black_border_mask
            flood_mask = np.zeros((h_img + 2, w_img + 2), np.uint8)  # FIX: fresh mask per seed
            if black_border_mask[numpy_row, numpy_col] == 1:
                cv2.floodFill(clean_pred, flood_mask, (cx, cy), 0)

        # FIX: morphological closing to fill small gaps left in thin roads
        kernel     = np.ones((3, 3), np.uint8)
        clean_pred = cv2.morphologyEx(clean_pred, cv2.MORPH_CLOSE, kernel)

        print("[SUCCESS] Border artifacts removed + morphological closing applied.")

        plt.figure(figsize=(15, 5))
        plt.subplot(1, 3, 1); plt.title('Original Satellite Image'); plt.imshow(sample_img);      plt.axis('off')
        plt.subplot(1, 3, 2); plt.title('Raw Model Output');         plt.imshow(binary_mask, cmap='gray'); plt.axis('off')
        plt.subplot(1, 3, 3); plt.title('Flood Fill + Morph Close'); plt.imshow(clean_pred, cmap='gray'); plt.axis('off')
        plt.tight_layout()
        plt.show()
    #####################################################################
    # STEP 7: NOVEL CONNECTIVITY METRIC -- FULL TEST SET EVALUATION
    #####################################################################
    print("\n" + "="*50)
    print(" CONNECTIVITY METRIC -- FULL TEST SET")
    print("="*50)

    def connectivity_score(pred_bin, true_bin):
        """Novel metric: ratio of connected components (GT / Pred).
        Near 1.0 = well-connected roads. << 1.0 = fragmented prediction."""
        _, n_pred = cv2.connectedComponents(pred_bin)
        _, n_true = cv2.connectedComponents(true_bin)
        return n_true / max(n_pred, 1)

    conn_scores, iou_np = [], []
    num_eval = min(len(test_x), 50)
    print("[INFO] Evaluating connectivity on " + str(num_eval) + " samples...")
    for idx in range(num_eval):
        t_img = read_image(test_x[idx])
        t_msk = read_mask(test_y[idx])
        if t_img is None or t_msk is None: continue
        t_pred = model.predict(np.expand_dims(t_img, 0), verbose=0)[0]
        pb = (t_pred.squeeze() > 0.5).astype(np.uint8)
        tb = (t_msk.squeeze() > 0.5).astype(np.uint8)
        inter = np.sum(pb * tb)
        union = np.sum(pb) + np.sum(tb) - inter
        iou_np.append(float((inter + 1e-6) / (union + 1e-6)))
        conn_scores.append(connectivity_score(pb, tb))

    print("\n-- Connectivity Analysis (" + str(len(conn_scores)) + " samples) --")
    print("  Mean IoU:          " + str(round(float(np.mean(iou_np)), 4)))
    print("  Mean Connectivity: " + str(round(float(np.mean(conn_scores)), 4)))
    print("  Std  Connectivity: " + str(round(float(np.std(conn_scores)), 4)))
